# bigcompute.science MCP Explorer

**22 tools for computational mathematics research.** Query experiments, cross-reference findings against arXiv/zbMATH/OEIS, search Lean 4/Mathlib, and explore open problems.

No API key needed. No setup. Just run the cells.

**Tip:** Use Gemini in the sidebar (or any AI assistant) to reason about the data. Ask it to explain the math, spot patterns, or suggest next experiments. Everything in this notebook is designed to be fed into an AI conversation.

- Website: [bigcompute.science](https://bigcompute.science)
- Audit Ledger: [bigcompute.science/verification](https://bigcompute.science/verification/)
- Source: [github.com/cahlen/idontknow](https://github.com/cahlen/idontknow)
- License: CC BY 4.0

## Setup

Run this cell to connect to the MCP server and mount all Hugging Face datasets.

In [ ]:
# === Setup: MCP Client + HF Datasets (run this first) ===
!pip install -q huggingface_hub datasets requests

import requests, json
from huggingface_hub import hf_hub_download, list_repo_files
from pathlib import Path

# --- MCP Client ---
MCP = "https://mcp.bigcompute.science/mcp"

requests.post(MCP, json={
    "jsonrpc": "2.0", "id": 0,
    "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "colab-explorer", "version": "2.0"}}
})

def mcp(tool, **kwargs):
    """Call any MCP tool. Returns parsed JSON."""
    r = requests.post(MCP, json={
        "jsonrpc": "2.0", "id": 1,
        "method": "tools/call",
        "params": {"name": tool, "arguments": kwargs}
    })
    text = r.json().get("result", {}).get("content", [{}])[0].get("text", "{}")
    try: return json.loads(text)
    except: return text

def pp(data):
    """Pretty print JSON."""
    print(json.dumps(data, indent=2) if isinstance(data, (dict, list)) else data)

# --- Mount HF Datasets ---
HF_DATASETS = {
    'zaremba': 'cahlen/zaremba-conjecture-data',
    'ramanujan': 'cahlen/ramanujan-machine-results',
    'kronecker': 'cahlen/kronecker-coefficients',
    'spectra': 'cahlen/continued-fraction-spectra',
}

DATA = {}
print('Mounting HF datasets...')
for name, repo in HF_DATASETS.items():
    files = list_repo_files(repo, repo_type='dataset')
    csvs = [f for f in files if f.endswith('.csv')]
    print(f'  {name}: {len(files)} files ({len(csvs)} CSVs)')
    DATA[name] = {'repo': repo, 'files': files, 'csvs': csvs}

def load_csv(dataset_name, filename):
    """Download and load a CSV from any HF dataset."""
    import pandas as pd
    repo = DATA[dataset_name]['repo']
    path = hf_hub_download(repo_id=repo, filename=filename, repo_type='dataset')
    return pd.read_csv(path)

print()
print('Connected to bigcompute.science MCP (22 tools)')
print('Datasets mounted. Use load_csv("zaremba", "density/density_all_subsets_n10_1e6.csv")')
print()
print('Tip: Ask Gemini in the sidebar to help analyze the data!')

## What's Available

### Experiments & Findings

In [ ]:
# All experiments
experiments = mcp('list_experiments')
print('EXPERIMENTS:')
for e in experiments:
    print(f"  [{e['status']:12s}] {e['title'][:65]}")

print()

# All findings with audit levels
findings = mcp('list_findings')
print('FINDINGS:')
for f in findings:
    cert = f.get('certification', {})
    level = cert.get('level', '?').upper()
    print(f"  [{level:6s}] {f['title'][:60]}")

### HF Dataset Files

Every dataset is available for direct download and analysis:

In [ ]:
# Browse available files in each dataset
for name, info in DATA.items():
    print(f"\n=== {name} ({info['repo']}) ===")
    for f in sorted(info['files'])[:15]:
        if f != '.gitattributes':
            print(f"  {f}")
    if len(info['files']) > 15:
        print(f"  ... and {len(info['files'])-15} more")

---
## Hands-On: Zaremba Density Phase Transition

Let's load the complete density sweep (all 1,023 subsets of {1,...,10}) and visualize the phase transition.

**Ask Gemini:** *"What patterns do you see in the density data? Why does digit 1 matter so much?"*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the complete density sweep
df = load_csv('zaremba', 'density/density_all_subsets_n10_1e6.csv')
print(f'Loaded {len(df)} subsets')
print(df.head())
print(f'\nColumns: {list(df.columns)}')

In [ ]:
# Visualize: density vs cardinality, colored by whether digit 1 is present
df['has_1'] = df['digits'].str.contains('1')

fig, ax = plt.subplots(figsize=(10, 6))

for has, label, color in [(True, 'Contains digit 1', '#e8c47a'), (False, 'No digit 1', '#5eead4')]:
    subset = df[df['has_1'] == has]
    ax.scatter(subset['cardinality'] + (0.1 if has else -0.1),
               subset['density'], alpha=0.3, s=10, c=color, label=label)

ax.set_xlabel('Cardinality |A|', fontsize=12)
ax.set_ylabel('Density (%)', fontsize=12)
ax.set_title('Zaremba Density by Digit Set Cardinality (N = 10^6)', fontsize=14)
ax.legend(fontsize=11)
ax.set_facecolor('#0b0d10')
fig.patch.set_facecolor('#0b0d10')
ax.tick_params(colors='#8a8580')
ax.xaxis.label.set_color('#c4c1bc')
ax.yaxis.label.set_color('#c4c1bc')
ax.title.set_color('#e8e6e3')
ax.spines['bottom'].set_color('#2a2e35')
ax.spines['left'].set_color('#2a2e35')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('\nDigit 1 advantage by cardinality:')
for card in range(1, 11):
    w = df[(df['cardinality']==card) & df['has_1']]['density']
    wo = df[(df['cardinality']==card) & ~df['has_1']]['density']
    if len(w) > 0 and len(wo) > 0:
        print(f'  |A|={card}: with 1 = {w.mean():.1f}%, without = {wo.mean():.1f}%, gap = {w.mean()-wo.mean():.1f}pp')

### The 27 Exceptions

These are the only integers (out of 10 billion tested) that have no coprime fraction with all CF partial quotients in {1,2,3}.

**Ask Gemini:** *"What's special about these 27 numbers? Why do they resist Zaremba representation?"*

In [ ]:
# The famous 27 exceptions
exc = mcp('get_zaremba_exceptions')
exceptions = exc['exceptions']

print(f'The {len(exceptions)} exceptions for A={{1,2,3}}:')
print(exceptions)
print(f'\nAll below {exc["all_below"]}')
print(f'Verified to {exc["verified_to"]}')

# Analyze their structure
print('\nFactorizations:')
from sympy import factorint
for d in exceptions:
    factors = factorint(d)
    fstr = ' x '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(factors.items()))
    print(f'  {d:>5d} = {fstr}')

print(f'\nHierarchy:')
for k, v in exc['hierarchy'].items():
    print(f'  {k}: {v}')

---
## Hands-On: Hausdorff Dimension Spectrum

Load the complete spectrum of Hausdorff dimensions for all 1,048,575 subsets of {1,...,20}.

**Ask Gemini:** *"Why does removing digit 1 from {1,...,20} cost more dimension than removing digits 2 through 20 combined?"*

In [ ]:
# Load the Hausdorff dimension spectrum
# Note: spectrum_n20.csv has commas inside the digits field, parse carefully
import csv

path = hf_hub_download('cahlen/continued-fraction-spectra',
                       'hausdorff-spectrum/spectrum_n20.csv', repo_type='dataset')

dimensions = []
with open(path) as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        mask = int(row[0])
        dim = float(row[-1])  # dimension is always last
        card = bin(mask).count('1')
        has_1 = mask & 1
        dimensions.append({'mask': mask, 'cardinality': card, 'dimension': dim, 'has_1': bool(has_1)})

df_h = pd.DataFrame(dimensions)
print(f'Loaded {len(df_h)} subsets')
print(f'Dimension range: [{df_h["dimension"].min():.6f}, {df_h["dimension"].max():.6f}]')
print(f'\nKey values:')
# E_{1,...,5}
mask_12345 = 0b11111  # bits 0-4
d5 = df_h[df_h['mask'] == mask_12345]['dimension'].values[0]
print(f'  dim_H(E_{{1,...,5}}) = {d5:.6f}  (Zaremba conjecture set)')
# E_{2,...,20}
mask_2to20 = sum(1 << i for i in range(1, 20))  # bits 1-19
d_no1 = df_h[df_h['mask'] == mask_2to20]['dimension'].values[0]
print(f'  dim_H(E_{{2,...,20}}) = {d_no1:.6f}  (19 digits without 1)')
print(f'  5 digits with 1 > 19 digits without 1:  {d5:.4f} > {d_no1:.4f}')

In [ ]:
# Visualize: dimension distribution by cardinality
fig, ax = plt.subplots(figsize=(10, 6))

for has, label, color in [(True, 'Contains digit 1', '#e8c47a'), (False, 'No digit 1', '#5eead4')]:
    subset = df_h[df_h['has_1'] == has]
    ax.scatter(subset['cardinality'] + (0.1 if has else -0.1),
               subset['dimension'], alpha=0.05, s=3, c=color, label=label)

ax.axhline(y=0.5, color='#ef4444', linestyle='--', alpha=0.5, label='delta = 1/2 threshold')
ax.set_xlabel('Cardinality |A|', fontsize=12)
ax.set_ylabel('Hausdorff Dimension', fontsize=12)
ax.set_title('Hausdorff Dimension Spectrum (all subsets of {1,...,20})', fontsize=14)
ax.legend(fontsize=10)
ax.set_facecolor('#0b0d10')
fig.patch.set_facecolor('#0b0d10')
ax.tick_params(colors='#8a8580')
ax.xaxis.label.set_color('#c4c1bc')
ax.yaxis.label.set_color('#c4c1bc')
ax.title.set_color('#e8e6e3')
for spine in ax.spines.values(): spine.set_color('#2a2e35')
plt.tight_layout()
plt.show()

---
## Cross-Reference: Verify Claims Against Literature

Pick any finding and check it against arXiv, zbMATH, OEIS, and more.

**Ask Gemini:** *"Does the Bourgain-Kontorovich paper support or contradict this finding? What does the zbMATH review say?"*

In [ ]:
# Verify any finding — change the slug to explore others
result = mcp('verify_finding', finding='zaremba-density')

print(f"Finding: {result['finding']['title']}")
print(f"Claim: {result['finding']['claim'][:200]}...")
print()

for src in ['arxiv', 'zbmath', 'oeis']:
    items = result['literature'].get(src, [])
    if items:
        print(f"{src.upper()} ({len(items)} results):")
        for p in items[:3]:
            name = p.get('title', p.get('name', p.get('id', '?')))
            year = p.get('year', '')
            print(f"  [{year}] {name[:70]}")
        print()

print('Guidance:')
for g in result.get('verification_guidance', []):
    print(f'  {g}')

---
## Search: arXiv, zbMATH, OEIS, LMFDB, Lean/Mathlib

Use these tools for your own research. Change the queries to explore anything.

**Ask Gemini:** *"Search for papers about [your topic] and summarize what's known."*

In [ ]:
# Search arXiv
papers = mcp('search_arxiv', query='Zaremba conjecture continued fractions',
             category='math.NT', max_results=5)
for p in papers:
    print(f"[{p['published']}] {p['title']}")
    print(f"  {p['url']}")
    print(f"  {p['summary'][:120]}...")
    print()

In [ ]:
# Search OEIS — try your own sequence!
seqs = mcp('lookup_oeis', query='6,20,28,38,42,54,96,150')
if isinstance(seqs, list) and seqs:
    for s in seqs[:3]:
        print(f"{s['id']}: {s['name']}")
        print(f"  Data: {s.get('data', '')[:60]}")
        print()
elif isinstance(seqs, str) and 'No' in seqs:
    print('Not in OEIS! This might be a NEW sequence.')
    print('The Zaremba 27-exception sequence is not in OEIS — we discovered it.')
else:
    print(seqs)

In [ ]:
# Search zbMATH (peer-reviewed math publications)
papers = mcp('search_zbmath', query='Kronecker coefficients symmetric group', max_results=3)
for p in papers.get('papers', []):
    print(f"[{p.get('year', '?')}] {p['title']}")
    print(f"  by {p.get('authors', '?')}")
    if p.get('review'):
        print(f"  Review: {p['review'][:150]}...")
    print()

In [ ]:
# Search Lean 4 / Mathlib
# Note: Loogle blocks most cloud IPs. Works from local machines.
results = mcp('search_mathlib', query='Nat.Prime')
if isinstance(results, dict) and 'results' in results:
    print(f"Found {results.get('count', '?')} results:")
    for r in results['results'][:5]:
        print(f"  {r['name']} : {r.get('type', '?')[:50]}")
else:
    # Loogle blocks cloud IPs — try direct, handle failure gracefully
    try:
        r = requests.get('https://loogle.lean-lang.org/json?q=Nat.Prime', timeout=5)
        data = r.json()
        print(f"Found {data.get('count', '?')} results (direct query):")
        for h in data.get('hits', [])[:5]:
            print(f"  {h['name']} : {h.get('type', '?')[:50]}")
    except:
        print('Loogle is not accessible from this environment.')
        print('To search Mathlib, visit: https://loogle.lean-lang.org/?q=Nat.Prime')
        print('Or run this notebook locally where Loogle is reachable.')


---
## Reproduce: Get CUDA Kernels & Commands

Every experiment has source code and exact reproduction commands.

In [ ]:
# Get CUDA kernel and commands for any experiment
kernel = mcp('get_cuda_kernel', experiment='zaremba-density')
pp(kernel)

print(f"\nTo reproduce locally:")
print(f"  git clone https://github.com/cahlen/idontknow")
print(f"  cd idontknow")
print(f"  {kernel.get('compile', 'N/A')}")
print(f"  {kernel.get('run', 'N/A')}")

In [ ]:
# === Compile & Run: Use your Colab GPU for real math ===
# Auto-detects your GPU and compiles the right binary
import subprocess, os

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                           capture_output=True, text=True)
    gpu_name, compute_cap = result.stdout.strip().split(', ')
    major, minor = compute_cap.split('.')
    sm = f'sm_{major}{minor}'
    print(f'GPU: {gpu_name} (compute {compute_cap} -> {sm})')
except:
    sm = 'sm_75'
    print(f'Using default {sm} (T4)')

# Clone repo if not already cloned
if not os.path.exists('idontknow'):
    !git clone https://github.com/cahlen/idontknow.git
%cd idontknow

# Compile all kernels
for binary, source, libs in [
    ('zaremba_density_gpu', 'scripts/experiments/zaremba-density/zaremba_density_gpu.cu', '-lm'),
    ('kronecker_gpu', 'scripts/experiments/kronecker-coefficients/kronecker_gpu.cu', '-lm'),
    ('ramanujan_gpu', 'scripts/experiments/ramanujan-machine/ramanujan_gpu.cu', '-lm'),
]:
    if os.path.exists(source):
        ret = os.system(f'nvcc -O3 -arch={sm} -o {binary} {source} {libs} 2>&1')
        print(f'  {binary}: {"OK" if ret == 0 else "FAILED"}')


In [ ]:
# === Quick Wins (ALL finish in seconds) ===

import os
os.makedirs('idontknow/scripts/experiments/zaremba-density/results', exist_ok=True)

print('='*50)
print('Verify Zaremba conjecture to 100,000...')
print('='*50)
!./zaremba_density_gpu 100000 1,2,3,4,5

print('\n' + '='*50)
print('Find the Zaremba exceptions for A={1,2,3}...')
print('='*50)
!./zaremba_density_gpu 10000 1,2,3

print('\nDone! Uncomment below to go bigger:')
# !./zaremba_density_gpu 1000000 1,2,3,4,5    # 10^6, ~10 sec
# !./zaremba_density_gpu 1000000000 1,2,8      # 10^9, ~2 min, NEW DATA


---
## What Should I Compute?

If you have GPU hardware, here are the highest-impact experiments:

In [ ]:
suggestions = mcp('suggest_experiment', gpu_hours=10, interest='any')
for s in suggestions.get('recommended', suggestions.get('all_suggestions', [])):
    print(f"#{s['priority']} {s['experiment']}")
    print(f"   GPU hours: ~{s.get('estimated_gpu_hours', s.get('gpu_hours', '?'))}")
    print(f"   Impact: {s['impact'][:80]}")
    print(f"   Command: {s.get('command', 'N/A')}")
    print()

---
## Reference: All 22 Tools

Every tool is called via `mcp('tool_name', param1=value1, ...)`

In [ ]:
r = requests.post(MCP, json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}})
tools = r.json()['result']['tools']
print(f"{len(tools)} tools:\n")
for t in tools:
    params = ', '.join(f"{k}=..." for k in t.get('inputSchema', {}).get('properties', {}).keys())
    print(f"  mcp('{t['name']}'{', ' + params if params else ''})")
    print(f"    {t['description'][:70]}")
    print()

---
## Submit Your Work

Found something interesting? The cell below generates a formatted review or finding that you can copy-paste directly into a GitHub pull request.

1. Run the cell below
2. Fill in the prompts
3. Copy the output
4. Go to [github.com/cahlen/idontknow](https://github.com/cahlen/idontknow) → Fork → Add file → Paste → Submit PR

**Ask Gemini:** *"Help me write a review of the [finding name] based on what we found in this notebook."*

In [ ]:
# === Generate a PR-ready review or finding ===
from datetime import datetime

print('What did you do?')
print('  1. Reviewed a finding (audit)')
print('  2. Discovered something new (finding)')
print('  3. Reproduced a computation (verification)')
choice = input('\nChoice (1/2/3): ').strip()

if choice == '1':
    slug = input('Finding slug (e.g. zaremba-density-phase-transition): ').strip()
    your_name = input('Your name or model name: ').strip()
    verdict = input('Verdict (VERIFIED / NEEDS_CLARIFICATION / DISPUTED): ').strip()
    assessment = input('Your assessment (1-2 sentences): ').strip()
    sources = input('Sources you checked (comma-separated): ').strip()
    
    review = {
        'finding_slug': slug,
        'verified_by': your_name,
        'verified_at': datetime.utcnow().isoformat() + 'Z',
        'sources_checked': [s.strip() for s in sources.split(',')],
        'assessment': assessment,
        'verdict': verdict,
    }
    
    print('\n' + '='*60)
    print(f'COPY THIS INTO: docs/verifications/{slug}_{your_name.replace(" ","-").lower()}.json')
    print('='*60)
    print(json.dumps(review, indent=2))
    print('='*60)
    print(f'\nPR title: verification: {slug}')
    print(f'PR to: github.com/cahlen/idontknow')

elif choice == '2':
    title = input('Finding title: ').strip()
    claim = input('What did you find? (1-2 sentences): ').strip()
    data = input('Key data points (comma-separated): ').strip()
    method = input('How did you compute this? ').strip()
    your_name = input('Your name: ').strip()
    
    print('\n' + '='*60)
    print('COPY THIS AS A NEW FINDING (Markdown):')
    print('='*60)
    print(f'# {title}')
    print(f'\n**Claim:** {claim}')
    print(f'\n**Data:** {data}')
    print(f'\n**Method:** {method}')
    print(f'\n**Computed by:** {your_name}')
    print(f'**Date:** {datetime.utcnow().strftime("%Y-%m-%d")}')
    print(f'\n*Not independently verified. Submit as PR to github.com/cahlen/idontknow.*')
    print('='*60)
    print(f'\nPR title: finding: {title[:50]}')

elif choice == '3':
    experiment = input('Experiment slug: ').strip()
    your_name = input('Your name: ').strip()
    hardware = input('Your hardware: ').strip()
    result = input('Did it match? (yes/no/partial): ').strip()
    notes = input('Notes: ').strip()
    
    print('\n' + '='*60)
    print('COPY THIS INTO YOUR PR DESCRIPTION:')
    print('='*60)
    print(f'## Reproduction: {experiment}')
    print(f'\n**Reproduced by:** {your_name}')
    print(f'**Hardware:** {hardware}')
    print(f'**Result matches:** {result}')
    print(f'**Notes:** {notes}')
    print(f'**Date:** {datetime.utcnow().strftime("%Y-%m-%d")}')
    print('='*60)
    print(f'\nPR title: reproduction: {experiment}')

else:
    print('Invalid choice')

---

**bigcompute.science** — Guerrilla Mathematics™

AI-audited, not peer-reviewed. All code and data open. CC BY 4.0.

**Want to contribute?**
- Run computations on your own GPU and [submit results](https://github.com/cahlen/idontknow/blob/main/AGENTS.md)
- [Submit an audit review](https://github.com/cahlen/idontknow/blob/main/docs/verifications/SCHEMA.md) of any finding
- Use the data in your own research (just cite us!)

**Ask Gemini:** *"Based on everything in this notebook, what's the most interesting open question? What should I compute next?"*